# Morphology R&D: padding, initialization, BIC threshold, jump skew, rejectors

This notebook compares morphology fit behavior across run-padding width and initialization strategies, and probes jump skew-gaussian + rejector-model options.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from malca.events import score_lightcurve
from malca.utils import read_lc_dat2, read_skypatrol_csv

plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
INPUT_RESULTS = Path('output/runs/latest/results/lc_events_results.parquet')
MAX_LIGHTCURVES = 80
RNG_SEED = 42

if INPUT_RESULTS.suffix.lower() in {'.parquet', '.pq'}:
    results_df = pd.read_parquet(INPUT_RESULTS)
else:
    results_df = pd.read_csv(INPUT_RESULTS)

if 'path' not in results_df.columns:
    raise KeyError('Input table must include a path column.')

results_df = results_df.dropna(subset=['path']).copy()
results_df['path'] = results_df['path'].astype(str)

# Prefer rows with at least one event branch active to focus R&D effort.
if {'dip_significant', 'jump_significant'}.issubset(results_df.columns):
    active = results_df[(results_df['dip_significant'].fillna(False)) | (results_df['jump_significant'].fillna(False))]
    pool = active if not active.empty else results_df
else:
    pool = results_df

sample_df = pool.sample(n=min(MAX_LIGHTCURVES, len(pool)), random_state=RNG_SEED) if len(pool) > 0 else pool
sample_df = sample_df.reset_index(drop=True)
print(f'Sampled {len(sample_df)} light curves from {len(results_df)} rows')
sample_df[['path']].head()

In [ ]:
def load_lightcurve(path_str: str) -> pd.DataFrame:
    p = Path(path_str).expanduser()
    if not p.exists():
        return pd.DataFrame()
    if p.suffix.lower() == '.csv':
        return read_skypatrol_csv(str(p))
    if p.suffix.lower() == '.dat2':
        dfg, dfv = read_lc_dat2(p.stem, str(p.parent))
        return pd.concat([dfg, dfv], ignore_index=True)
    return pd.DataFrame()

def best_run_summary(branch: dict) -> dict:
    runs = branch.get('run_summaries', []) or []
    if not runs:
        return {'morphology': 'none', 'delta_bic_null': np.nan, 'rejector_model': 'none', 'rejector_delta_bic': np.nan}
    best = max(runs, key=lambda r: float(r.get('run_max', -np.inf)))
    return {
        'morphology': best.get('morphology', 'none'),
        'delta_bic_null': best.get('delta_bic_null', np.nan),
        'rejector_model': best.get('rejector_model', 'none'),
        'rejector_delta_bic': best.get('rejector_delta_bic', np.nan),
    }

def evaluate_config(df_lc: pd.DataFrame, *, pad_points: int, init_strategy: str, delta_bic_threshold: float, jump_skew: bool, rejectors: bool) -> dict:
    res = score_lightcurve(
        df_lc,
        trigger_mode='logbf',
        logbf_threshold_dip=5.0,
        logbf_threshold_jump=5.0,
        morph_pad_points=pad_points,
        morph_init_strategy=init_strategy,
        morph_delta_bic_threshold=delta_bic_threshold,
        include_jump_skew_gaussian=jump_skew,
        enable_rejector_models=rejectors,
    )
    dip_best = best_run_summary(res['dip'])
    jump_best = best_run_summary(res['jump'])
    return {
        'dip_sig': bool(res['dip'].get('significant', False)),
        'jump_sig': bool(res['jump'].get('significant', False)),
        'dip_morph': str(dip_best['morphology']),
        'jump_morph': str(jump_best['morphology']),
        'dip_delta_bic': float(dip_best['delta_bic_null']),
        'jump_delta_bic': float(jump_best['delta_bic_null']),
        'dip_rejector_model': str(dip_best['rejector_model']),
        'jump_rejector_model': str(jump_best['rejector_model']),
        'dip_rejector_delta_bic': float(dip_best['rejector_delta_bic']),
        'jump_rejector_delta_bic': float(jump_best['rejector_delta_bic']),
    }

In [ ]:
grid_rows = []
for pad in [3, 5, 8, 12]:
    for init in ['peak_abs', 'run_peak', 'run_midpoint']:
        for bic_thr in [6.0, 10.0, 14.0]:
            for jump_skew in [False, True]:
                # keep rejector sweep separate to limit run time
                grid_rows.append((pad, init, bic_thr, jump_skew, False))

records = []
for _, row in sample_df.iterrows():
    lc = load_lightcurve(row['path'])
    if lc.empty:
        continue
    for (pad, init, bic_thr, jump_skew, rejectors) in grid_rows:
        try:
            out = evaluate_config(
                lc,
                pad_points=pad,
                init_strategy=init,
                delta_bic_threshold=bic_thr,
                jump_skew=jump_skew,
                rejectors=rejectors,
            )
            out.update({
                'path': row['path'],
                'pad_points': pad,
                'init_strategy': init,
                'delta_bic_threshold': bic_thr,
                'jump_skew': jump_skew,
                'rejectors': rejectors,
            })
            records.append(out)
        except Exception:
            continue

eval_df = pd.DataFrame(records)
print(f'Evaluated rows: {len(eval_df):,}')
eval_df.head()

In [ ]:
if not eval_df.empty:
    summary = eval_df.groupby(['pad_points', 'init_strategy', 'delta_bic_threshold', 'jump_skew']).agg(
        n=('path', 'count'),
        dip_sig_rate=('dip_sig', 'mean'),
        jump_sig_rate=('jump_sig', 'mean'),
        dip_delta_bic_med=('dip_delta_bic', 'median'),
        jump_delta_bic_med=('jump_delta_bic', 'median'),
        jump_skew_fraction=('jump_morph', lambda s: float((s == 'skew_gaussian').mean())),
    ).reset_index()
    display(summary.sort_values(['jump_sig_rate', 'jump_skew_fraction'], ascending=False).head(20))
else:
    print('No evaluation rows available')

In [ ]:
# Rejector-model probe (subset configs for speed)
rejector_records = []
for _, row in sample_df.head(min(40, len(sample_df))).iterrows():
    lc = load_lightcurve(row['path'])
    if lc.empty:
        continue
    try:
        out = evaluate_config(
            lc,
            pad_points=8,
            init_strategy='run_peak',
            delta_bic_threshold=10.0,
            jump_skew=True,
            rejectors=True,
        )
        out['path'] = row['path']
        rejector_records.append(out)
    except Exception:
        continue

rejector_df = pd.DataFrame(rejector_records)
print(f'Rejector-evaluated rows: {len(rejector_df):,}')
if not rejector_df.empty:
    display(rejector_df[['dip_rejector_model', 'jump_rejector_model']].apply(pd.Series.value_counts).fillna(0))
    display(rejector_df[['dip_rejector_delta_bic', 'jump_rejector_delta_bic']].describe())